In [1]:
import pandas as pd
from pathlib import Path
import os
import random
import numpy as np
import cv2
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader , WeightedRandomSampler
import keras
from torchvision.models.video import mvit_v2_s , MViT_V2_S_Weights
import torch.nn.functional as F
import torch
import torch.nn as nn
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score , roc_auc_score
from pathlib import Path
import math
from sklearn.metrics import  classification_report, confusion_matrix
from torch.optim.swa_utils import AveragedModel, get_ema_multi_avg_fn



c:\Users\Toranj\Desktop\AIM\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

os.environ["KERAS_BACKEND"] = "torch"

print("Keras version:", keras.__version__)
print("Keras backend:", keras.backend.backend())
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



Keras version: 3.15.1
Keras backend: tensorflow
PyTorch version: 2.14.1+cu130
CUDA available: True
GPU: NVIDIA GeForce RTX 5050 Laptop GPU


In [ ]:
"""
log_path = "nexar_data/processed_frames_matched2/clean_balanced_log.csv"
data = pd.read_csv(log_path)

data = data[data["status"].isin(["ok", "ok_existing"])].copy()
data = data.dropna(subset=["clip_path", "label"])
data["label"] = data["label"].astype(int)

train_df, temp_df = train_test_split(data , test_size=0.30 , random_state=42 , stratify=data["label"])
val_df, test_df = train_test_split(temp_df , test_size=0.50 , random_state=42 , stratify=temp_df["label"])

train_df.to_csv("nexar_data/train.csv", index=False)
val_df.to_csv("nexar_data/val.csv", index=False)
test_df.to_csv("nexar_data/test.csv", index=False)

for name, part in [("Train", train_df) , ("Validation", val_df) , ("Test", test_df)]:
    print(f"\n{name}: {len(part)} videos")
    print(part["label"].value_counts().sort_index())
"""


Train: 789 videos
label
0    394
1    395
Name: count, dtype: int64

Validation: 169 videos
label
0    85
1    84
Name: count, dtype: int64

Test: 170 videos
label
0    85
1    85
Name: count, dtype: int64


In [ ]:
OUTPUT_DIR = Path("nexar_data/processed_frames_matched2")

train_df = pd.read_csv(OUTPUT_DIR / "train_manifest.csv")
val_df   = pd.read_csv(OUTPUT_DIR / "val_manifest.csv")
train_df["target_label"] = train_df["target_label"].astype(int)
val_df["target_label"] = val_df["target_label"].astype(int)


train_df["sample_weight"] = train_df["sample_weight"].astype(float)

print("Training:", len(train_df))
print("Validation:", len(val_df))

print("\nTraining labels:")
print(train_df["target_label"].value_counts().sort_index())

print("\nValidation labels:")
print(val_df["target_label"].value_counts().sort_index())

Training: 1091
Validation: 273

Training labels:
target_label
0    643
1    448
Name: count, dtype: int64

Validation labels:
target_label
0    161
1    112
Name: count, dtype: int64


In [ ]:
print(train_df.groupby("temporal_group").agg(clips=("target_label", "count"), weight=("sample_weight", "first")))
s
weighted_mass = train_df.groupby("temporal_group")["sample_weight"].sum()

print("\nExpected sampling distribution:")
print((weighted_mass / weighted_mass.sum() * 100).round(2))

                       clips  weight
temporal_group                      
alert_to_event           448    10.0
negative                 491     1.0
positive_before_alert    152     2.0

Expected sampling distribution:
temporal_group
alert_to_event           84.93
negative                  9.31
positive_before_alert     5.76
Name: sample_weight, dtype: float64


In [ ]:

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(44)
 
class NexarDatasetAug(Dataset):
 
    def __init__(self, dataframe, train=False, use_flip=True):
        self.data = dataframe.reset_index(drop=True)
        self.train = train
        self.use_flip = use_flip
        self.mean = torch.tensor([0.45, 0.45, 0.45]).view(3, 1, 1, 1)
        self.std = torch.tensor([0.225, 0.225, 0.225]).view(3, 1, 1, 1)
 
    def __len__(self):
        return len(self.data)
 
    def _augment(self, x):
        C, T, H, W = x.shape

        # flip
        if self.use_flip and random.random() < 0.5:
            x = torch.flip(x, dims=[3])

        # random resized crop 
        if random.random() < 0.5:
            scale = random.uniform(0.85, 1.0)
            ratio = random.uniform(0.9, 1.1)
            area = scale * H * W
            
            w = min(W, int(round(math.sqrt(area * ratio))))
            h = min(H, int(round(math.sqrt(area / ratio))))

            top = random.randint(0, H - h)
            left = random.randint(0, W - w)

            x = x[:, :, top:top+h, left:left+w]
            x = F.interpolate(x.permute(1, 0, 2, 3), size=(H, W), mode="bilinear", align_corners=False).permute(1, 0, 2, 3)


        # brightness
        if random.random() < 0.5:
            brightness = random.uniform(0.85, 1.15)
            x = x * brightness

        # contrast
        if random.random() < 0.5:
            contrast = random.uniform(0.85, 1.15)
            mean = x.mean()
            x = (x - mean) * contrast + mean

        # saturation
        if random.random() < 0.3:
            saturation = random.uniform(0.8, 1.2)
            gray = (0.299 * x[0:1] + 0.587 * x[1:2] + 0.114 * x[2:3])
            x = gray + saturation * (x - gray)


        # 5. gaussian noise
        #if random.random() < 0.3:
        #    noise = torch.randn_like(x) * random.uniform(0.005, 0.02)
        #    x = x + noise

        return x.clamp(0.0, 1.0)

 
    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        frames = torch.from_numpy(np.load(row["clip_path"]).copy())
        frames = frames.permute(3, 0, 1, 2).float() / 255.0
        if self.train:
            frames = self._augment(frames)w
        frames = (frames - self.mean) / self.std

        # Use temporal target
        label = torch.tensor(int(row["target_label"]), dtype=torch.long)

        return frames, label  
 
 

def set_backbone_trainable(model, trainable):
    for p in model.backbone.parameters():
        p.requires_grad = trainable

    for p in model.temporal_pool.parameters():
        p.requires_grad = True

    for p in model.classifier.parameters():
        p.requires_grad = True
 


def build_optimizer(model, lr_head=2e-5, lr_top=5e-6, lr_low=1e-6, weight_decay=0.01, top_n=4):
    blocks = list(model.backbone.blocks)
    head_params = list(model.temporal_pool.parameters()) + list(model.classifier.parameters())
    top_params = [p for block in blocks[-top_n:] for p in block.parameters()]

    used = {
        id(p) for p in head_params + top_params
    }

    low_params = [p for p in model.backbone.parameters() if id(p) not in used]

    return AdamW([
        {"params": low_params, "lr": lr_low},
        {"params": top_params, "lr": lr_top},
        {"params": head_params, "lr": lr_head},
    ], 
    
    weight_decay=weight_decay)


# creating datasets
train_dataset = NexarDatasetAug(train_df, train=True, use_flip=False)
val_dataset = NexarDatasetAug(val_df, train=False, use_flip=False)

# assigning sampling weights
train_df["sample_weight"] = train_df["temporal_group"].map({
    "negative": 1.0,
    "positive_before_alert": 2.0,
    "alert_to_event": 3.0
})

# createing weighted sampler
sample_weights = torch.as_tensor(train_df["sample_weight"].to_numpy(), dtype=torch.double)
train_sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(train_dataset), replacement=True)

# creating DataLoaders
train_loader = DataLoader(train_dataset, batch_size=4, sampler=train_sampler, num_workers=0, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())




videos, labels = next(iter(train_loader))
print("Video batch:", videos.shape)
print("Labels:", labels)


C:\Users\Toranj\AppData\Local\Temp\ipykernel_12752\1715361333.py:147: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  sample_weights = torch.as_tensor(


Video batch: torch.Size([4, 3, 16, 224, 224])
Labels: tensor([0, 1, 1, 0])


In [6]:
videos, labels = next(iter(train_loader))

print("Video batch:", videos.shape)
print("Labels:", labels)
print("Label distribution:", torch.bincount(labels, minlength=2))

Video batch: torch.Size([4, 3, 16, 224, 224])
Labels: tensor([0, 1, 1, 0])
Label distribution: tensor([2, 2])


In [7]:
from collections import Counter

sampled_labels = []

for _, labels in train_loader:
    sampled_labels.extend(labels.tolist())

counts = Counter(sampled_labels)
total = len(sampled_labels)

print("Sampled label distribution:")
for label in [0, 1]:
    count = counts[label]
    print(
        f"Label {label}: {count} "
        f"({count / total * 100:.2f}%)"
    )

Sampled label distribution:
Label 0: 401 (36.76%)
Label 1: 690 (63.24%)


In [ ]:
# temporal attention / pool 

class MultiScaleTemporalPool(nn.Module):

    def __init__(self, feature_dim=768, num_heads=8, num_layers=2, dropout=0.2, num_frames=16):
        super().__init__()

        # spatial feature projection
        # average pooling + max pooling
        self.spatial_projection = nn.Sequential(
            nn.Linear(feature_dim * 2, feature_dim),
            nn.LayerNorm(feature_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        # learning temporal 
        self.pos_embedding = nn.Parameter(torch.zeros(1, num_frames, feature_dim))
        nn.init.trunc_normal_(self.pos_embedding, std=0.02)

        # temporal transform
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=feature_dim,
            nhead=num_heads,
            dim_feedforward=feature_dim * 2,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )

        self.temporal_transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.transformer_norm = nn.LayerNorm(feature_dim)



        # multy temporal cnn
        branch_dim = feature_dim // 3
        self.conv3 = nn.Sequential(
            nn.Conv1d(feature_dim, branch_dim, kernel_size=3, padding=1),
            nn.BatchNorm1d(branch_dim),
            nn.GELU()
        )

        self.conv5 = nn.Sequential(
            nn.Conv1d(feature_dim, branch_dim, kernel_size=5, padding=2),
            nn.BatchNorm1d(branch_dim),
            nn.GELU()
        )

        self.conv7 = nn.Sequential(
            nn.Conv1d(feature_dim, branch_dim, kernel_size=7, padding=3),
            nn.BatchNorm1d(branch_dim),
            nn.GELU()
        )

        self.multiscale_fusion = nn.Sequential(
            nn.Conv1d(branch_dim * 3, feature_dim, kernel_size=1),
            nn.BatchNorm1d(feature_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        # attention
        self.transformer_attention = nn.Sequential(
            nn.Linear(feature_dim, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )

        self.multiscale_attention = nn.Sequential(
            nn.Linear(feature_dim, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )

        # fuse transform + cnn representations
        self.attention_fusion = nn.Sequential(
            nn.Linear(feature_dim * 2, feature_dim),
            nn.LayerNorm(feature_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )

    def attention_pool(self, x, attention):
        weights = torch.softmax(attention(x), dim=1)
        return torch.sum(x * weights, dim=1)

    def forward(self, patch_tokens):

        # Input: [B, T, H, W, C]
        B, T, H, W, C = patch_tokens.shape

        avg_features = patch_tokens.mean(dim=(2, 3))
        max_features = patch_tokens.amax(dim=(2, 3))

        x = torch.cat([avg_features, max_features], dim=-1)

        # [B, T, 2C] -> [B, T, C]
        x = self.spatial_projection(x)


        # transformer branch
        transformer_input = x + self.pos_embedding[:, :T, :]
        

        transformer_features = self.temporal_transformer(transformer_input)
        transformer_features = self.transformer_norm(transformer_features)
        transformer_pooled = self.attention_pool(transformer_features, self.transformer_attention)

        # cnn branch
        conv_input = x.transpose(1, 2)

        conv3 = self.conv3(conv_input)
        conv5 = self.conv5(conv_input)
        conv7 = self.conv7(conv_input)

        multiscale_features = torch.cat([conv3, conv5, conv7], dim=1)
        multiscale_features = self.multiscale_fusion(multiscale_features)
        multiscale_features = multiscale_features.transpose(1, 2)
        multiscale_pooled = self.attention_pool(multiscale_features, self.multiscale_attention)

    

        attention_features = self.attention_fusion(torch.cat([transformer_pooled,multiscale_pooled], dim=1))

        # temporla statistics
        temporal_mean = x.mean(dim=1)
        temporal_max = x.amax(dim=1)

        # three 768-dimensional vectors
        return attention_features, temporal_mean, temporal_max
        




    

class NexarCollisionModel(nn.Module):

    def __init__(self, num_classes=2, pretrained=True, dropout=0.4):
        super().__init__()

        weights = MViT_V2_S_Weights.DEFAULT if pretrained else None
        
        self.backbone = mvit_v2_s(weights=weights)

        in_features = self.backbone.head[1].in_features

        # removeing original classification head
        self.backbone.head = nn.Identity()

        # multi temporal pooling
        self.temporal_pool = MultiScaleTemporalPool(
            feature_dim=768,
            num_heads=8,
            num_layers=2,
            dropout=0.2,
            num_frames=16
        )

    
        fusion_dim = in_features * 4

        self.classifier = nn.Sequential(
            nn.LayerNorm(fusion_dim),

            nn.Linear(fusion_dim, 512),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(512, 256),
            nn.GELU(),
            nn.Dropout(0.3),

            nn.Linear(256, num_classes)
        )

    def forward(self, x):

        # Input: [B, C, T, H, W]

        x = self.backbone.conv_proj(x)
        x = x.flatten(2).transpose(1, 2)

        x = self.backbone.pos_encoding(x)

        thw = (
            self.backbone.pos_encoding.temporal_size,
            *self.backbone.pos_encoding.spatial_size
        )

        # MViT transformer blocks
        for block in self.backbone.blocks:
            x, thw = block(x, thw)

        x = self.backbone.norm(x)

        cls_token = x[:, 0]
        patch_tokens = x[:, 1:]

        B = x.shape[0]
        T, H, W = thw
        C = x.shape[-1]

        # [B, T*H*W, C] -> [B, T, H, W, C]
        patch_tokens = patch_tokens.reshape(B, T, H, W, C)

        
        attention_features, temporal_mean, temporal_max = self.temporal_pool(patch_tokens)

        # Feature fusion
        features = torch.cat(
            [
                cls_token,
                attention_features,
                temporal_mean,
                temporal_max
            ],
            dim=1
        )

        return self.classifier(features)



model = NexarCollisionModel(num_classes=2, pretrained=True).to(device)

print(model)

C:\Users\Toranj\AppData\Local\Temp\ipykernel_12752\3452887440.py:50: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.temporal_transformer = nn.TransformerEncoder(


NexarCollisionModel(
  (backbone): MViT(
    (conv_proj): Conv3d(3, 96, kernel_size=(3, 7, 7), stride=(2, 4, 4), padding=(1, 3, 3))
    (pos_encoding): PositionalEncoding()
    (blocks): ModuleList(
      (0): MultiscaleBlock(
        (norm1): LayerNorm((96,), eps=1e-06, elementwise_affine=True, bias=True)
        (norm2): LayerNorm((96,), eps=1e-06, elementwise_affine=True, bias=True)
        (attn): MultiscaleAttention(
          (qkv): Linear(in_features=96, out_features=288, bias=True)
          (project): Sequential(
            (0): Linear(in_features=96, out_features=96, bias=True)
          )
          (pool_q): Pool(
            (pool): Conv3d(96, 96, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), groups=96, bias=False)
            (norm_act): Sequential(
              (0): LayerNorm((96,), eps=1e-06, elementwise_affine=True, bias=True)
            )
          )
          (pool_k): Pool(
            (pool): Conv3d(96, 96, kernel_size=(3, 3, 3), stride=(1, 8, 8), p

In [8]:

# loss function / optimizer / schedule and scaler 
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
val_criterion = nn.CrossEntropyLoss()   
optimizer = build_optimizer(model)
warmup = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, total_iters=1)
cosine = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=14, eta_min=1e-7)
scheduler = torch.optim.lr_scheduler.SequentialLR(optimizer, [warmup, cosine], milestones=[1])
scaler = torch.amp.GradScaler("cuda" , enabled=(device.type == "cuda"))
checkpoint_path = Path("nexar_data/best_mvit_collision.pth")
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)


In [9]:
# initializing constant 
num_epochs = 15
patience = 3
UNFREEZE_EPOCH = 5
best_val_auc = 0.0
epochs_without_improvement = 0
history = {
    "train_loss": [], 
    "val_loss": [],
    "train_f1": [], 
    "val_f1": [],
    "train_acc": [], 
    "val_acc": [],
    "val_auc": [],                                
}



# stopping backbone trainable to ensure model wont overfitt early 
set_backbone_trainable(model, False)  

ema = AveragedModel(model, multi_avg_fn=get_ema_multi_avg_fn(0.99))
# running model on training data and validation data 
for epoch in range(num_epochs):
 
    if epoch == UNFREEZE_EPOCH:
        # unpausing it after 3 epoch 
        set_backbone_trainable(model, True)
        print("    Backbone unfrozen")
 
    # training
    model.train()
    train_loss_sum = 0.0
    train_count = 0
    train_preds = []
    train_targets = []
 
    for videos, labels in train_loader:
        videos = videos.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
 
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=(device.type == "cuda")):
            outputs = model(videos)
            loss = criterion(outputs, labels)
 
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        ema.update_parameters(model)
 
        batch_size = labels.size(0)
        train_loss_sum += loss.item() * batch_size
        train_count += batch_size
 
        predictions = outputs.argmax(dim=1)
 
        train_preds.extend(predictions.detach().cpu().tolist())
        train_targets.extend(labels.cpu().tolist())
 
    train_loss = train_loss_sum / train_count
    train_acc = accuracy_score(train_targets, train_preds)
    train_f1 = f1_score(train_targets, train_preds, average="macro", zero_division=0)
 
    # validation
    model.eval()
 
    val_loss_sum = 0.0
    val_count = 0
    val_preds = []
    val_targets = []
    val_probs = []                           
 
    with torch.no_grad():
        for videos, labels in val_loader:
            videos = videos.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
 
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=(device.type == "cuda")):
                outputs = model(videos)
                loss = val_criterion(outputs, labels)     
 
            batch_size = labels.size(0)
            val_loss_sum += loss.item() * batch_size
            val_count += batch_size
 
            predictions = outputs.argmax(dim=1)
 
            val_preds.extend(predictions.cpu().tolist())
            val_targets.extend(labels.cpu().tolist())
            val_probs.extend(torch.softmax(outputs.float(), dim=1)[:, 1].cpu().tolist())  
 
    val_loss = val_loss_sum / val_count
    val_acc = accuracy_score(val_targets, val_preds)
    val_f1 = f1_score(val_targets, val_preds, average="macro", zero_division=0)
    val_precision = precision_score(val_targets, val_preds, average="macro", zero_division=0)
    val_recall = recall_score(val_targets, val_preds, average="macro", zero_division=0)
    val_auc = roc_auc_score(val_targets, val_probs)                                         
 
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_f1"].append(train_f1)
    history["val_f1"].append(val_f1)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)
    history["val_auc"].append(val_auc)
 
    # updating learning rate
    scheduler.step()


    # logggggs 
    print(
        f"Epoch [{epoch + 1:02d}/{num_epochs}] "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Train Macro F1: {train_f1:.4f} | "
        f"Val Macro F1: {val_f1:.4f} | "
        f"Val AUC: {val_auc:.4f}")
 
    print(
        f"    Val Macro Precision: {val_precision:.4f} | "
        f"Val Macro Recall: {val_recall:.4f} | "
        f"LR backbone: {optimizer.param_groups[0]['lr']:.2e} | "
        f"LR head: {optimizer.param_groups[1]['lr']:.2e}"
    )
 
    # saving best model
    if val_auc > best_val_auc + 1e-4:
        best_val_auc = val_auc
        epochs_without_improvement = 0

        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_val_auc": best_val_auc,
                "val_loss": val_loss,
                "val_f1": val_f1,
                "class_names": ["no_collision", "collision"]
            },
            checkpoint_path
        )

        print(f"    Saved best model: {checkpoint_path}")

    elif epoch >= UNFREEZE_EPOCH:
        epochs_without_improvement += 1
 
    if epochs_without_improvement >= patience:
        print("Early stopping.")
        break
 
best = torch.load(checkpoint_path, map_location=device)


# loggggs 
print(f"\nBest val AUC: {best['best_val_auc']:.4f} (epoch {best['epoch']}) | "
      f"val loss: {best['val_loss']:.4f} | val F1: {best['val_f1']:.4f}")
print(f"Checkpoint saved at: {checkpoint_path}")


Epoch [01/15] Train Loss: 0.6711 | Val Loss: 0.7509 | Train Acc: 0.6169 | Val Acc: 0.4103 | Train Macro F1: 0.4130 | Val Macro F1: 0.2909 | Val AUC: 0.6542
    Val Macro Precision: 0.2051 | Val Macro Recall: 0.5000 | LR backbone: 1.00e-06 | LR head: 5.00e-06
    Saved best model: nexar_data\best_mvit_collision.pth
Epoch [02/15] Train Loss: 0.6670 | Val Loss: 0.7224 | Train Acc: 0.6141 | Val Acc: 0.4103 | Train Macro F1: 0.4747 | Val Macro F1: 0.2909 | Val AUC: 0.6978
    Val Macro Precision: 0.2051 | Val Macro Recall: 0.5000 | LR backbone: 9.89e-07 | LR head: 4.94e-06
    Saved best model: nexar_data\best_mvit_collision.pth
Epoch [03/15] Train Loss: 0.6598 | Val Loss: 0.6968 | Train Acc: 0.6398 | Val Acc: 0.4982 | Train Macro F1: 0.5032 | Val Macro F1: 0.4629 | Val AUC: 0.7019
    Val Macro Precision: 0.6158 | Val Macro Recall: 0.5623 | LR backbone: 9.55e-07 | LR head: 4.76e-06
    Saved best model: nexar_data\best_mvit_collision.pth
Epoch [04/15] Train Loss: 0.6525 | Val Loss: 0.7734 

KeyboardInterrupt: 

In [ ]:
checkpoint = torch.load("nexar_data/best_mvit_collision.pth", map_location=device, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()


def get_probs(model, loader):
    probs, labels = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device, non_blocking=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device.type == "cuda")):
                logits = model(x)
            probs.extend(torch.softmax(logits.float(), dim=1)[:, 1].cpu().numpy())
            labels.extend(y.numpy())
    return np.array(probs), np.array(labels)


probs, labels = get_probs(model, val_loader)

ths = np.arange(0.20, 0.91, 0.01)

best_th = max(
    ths,
    key=lambda t: f1_score(
        labels,
        (probs >= t).astype(int),
        average="macro"
    )
)

#### logggggs
print("Best validation threshold:", round(best_th, 2))

preds_05 = (probs >= 0.5).astype(int)

print("\nResults at threshold 0.5:")
print("Macro F1:", f1_score(labels, preds_05, average="macro"))
print("AUC:", roc_auc_score(labels, probs))
print(confusion_matrix(labels, preds_05))

preds_best = (probs >= best_th).astype(int)

print("\nResults at optimized threshold:")
print("Macro F1:", f1_score(labels, preds_best, average="macro"))
print("Precision:", precision_score(labels, preds_best, average="macro"))
print("Recall:", recall_score(labels, preds_best, average="macro"))
print(confusion_matrix(labels, preds_best))

print("\nClassification Report:")
print(classification_report(labels, preds_best, digits=4))

Best validation threshold: 0.65

Results at threshold 0.5:
Macro F1: 0.6626370083816893
AUC: 0.7492513309671694
[[95 66]
 [26 86]]

Results at optimized threshold:
Macro F1: 0.7208549857829198
Precision: 0.7238095238095238
Recall: 0.718944099378882
[[128  33]
 [ 40  72]]

Classification Report:
              precision    recall  f1-score   support

           0     0.7619    0.7950    0.7781       161
           1     0.6857    0.6429    0.6636       112

    accuracy                         0.7326       273
   macro avg     0.7238    0.7189    0.7209       273
weighted avg     0.7306    0.7326    0.7311       273



In [20]:
rng = np.random.default_rng(42)
aucs, f1s = [], []
for _ in range(1000):
    idx = rng.integers(0, len(labels), len(labels))
    if len(np.unique(labels[idx])) < 2: continue
    aucs.append(roc_auc_score(labels[idx], probs[idx]))
    f1s.append(f1_score(labels[idx], preds_best[idx], average="macro"))

print(f"Test AUC : {roc_auc_score(labels, probs):.4f} "
      f"[95% CI {np.percentile(aucs,2.5):.4f}, {np.percentile(aucs,97.5):.4f}]")
print(f"Test F1  : {f1_score(labels, preds_best, average='macro'):.4f} "
      f"[95% CI {np.percentile(f1s,2.5):.4f}, {np.percentile(f1s,97.5):.4f}]")

Test AUC : 0.7571 [95% CI 0.6953, 0.8165]
Test F1  : 0.7131 [95% CI 0.6542, 0.7682]


In [21]:


for threshold in [0.50, 0.55, 0.60, 0.65,
                  0.70, 0.75, 0.79, 0.85]:

    preds = (probs >= threshold).astype(int)

    print(
        f"Threshold: {threshold:.2f} | "
        f"Macro F1: {f1_score(labels, preds, average='macro'):.4f} | "
        f"Precision: {precision_score(labels, preds, average='macro'):.4f} | "
        f"Recall: {recall_score(labels, preds, average='macro'):.4f}"
    )

Threshold: 0.50 | Macro F1: 0.6553 | Precision: 0.6680 | Recall: 0.6714
Threshold: 0.55 | Macro F1: 0.6645 | Precision: 0.6684 | Recall: 0.6739
Threshold: 0.60 | Macro F1: 0.6940 | Precision: 0.6930 | Recall: 0.6978
Threshold: 0.65 | Macro F1: 0.7073 | Precision: 0.7083 | Recall: 0.7065
Threshold: 0.70 | Macro F1: 0.7026 | Precision: 0.7152 | Recall: 0.6988
Threshold: 0.75 | Macro F1: 0.6901 | Precision: 0.7284 | Recall: 0.6865
Threshold: 0.79 | Macro F1: 0.6605 | Precision: 0.7284 | Recall: 0.6619
Threshold: 0.85 | Macro F1: 0.5850 | Precision: 0.7632 | Recall: 0.6126


In [13]:


#### test section for checking data leakage / not part of the model / let them be as it 
def static_frame(x):  
    return x[:, :, :1].expand(-1, -1, x.shape[2], -1, -1).contiguous()

def borders_only(x): 
    m = torch.ones_like(x)
    h, w = x.shape[-2:]
    m[..., h // 4: 3 * h // 4, w // 4: 3 * w // 4] = 0
    return x * m

def center_only(x): 
    m = torch.zeros_like(x)
    h, w = x.shape[-2:]
    m[..., h // 4: 3 * h // 4, w // 4: 3 * w // 4] = 1
    return x * m

def shuffle_time(x):  
    return x[:, :, torch.randperm(x.shape[2])]

@torch.no_grad()
def macro_f1(model, loader, device, transform=lambda x: x):
    model.eval()
    preds, targets = [], []
    for videos, labels in loader:
        out = model(transform(videos).to(device))
        preds += out.argmax(1).cpu().tolist()
        targets += labels.tolist()
    return f1_score(targets, preds, average="macro", zero_division=0)

def part_c(model, val_loader, device):
    variants = {
        "normal": lambda x: x,
        "static frame": static_frame,
        "shuffled time": shuffle_time,
        "borders only": borders_only,
        "center only": center_only,
    }
    for name, fn in variants.items():
        print(f"{name:14s} val macro-F1 = {macro_f1(model, val_loader, device, fn):.3f}")

In [14]:
ckpt = torch.load("nexar_data/best_mvit_collision.pth", map_location=device)
model = NexarCollisionModel(num_classes=2, pretrained=False).to(device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

val_loader = DataLoader(NexarDatasetAug(val_df), batch_size=4, shuffle=False, num_workers=0)
part_c(model, val_loader, device)

normal         val macro-F1 = 0.641
static frame   val macro-F1 = 0.460
shuffled time  val macro-F1 = 0.572
borders only   val macro-F1 = 0.671
center only    val macro-F1 = 0.722


In [15]:


def keep_strip(top=0.0, bottom=0.0, sides=0.0):
    def fn(x):  
        H, W = x.shape[-2:]
        m = torch.zeros_like(x)
        if top:
            m[..., : int(H * top), :] = 1
        if bottom:
            m[..., H - int(H * bottom):, :] = 1
        if sides:
            m[..., :, : int(W * sides)] = 1
            m[..., :, W - int(W * sides):] = 1
        return x * m
    return fn

@torch.no_grad()
def predict(model, loader, device, transform=lambda x: x):
    model.eval()
    probs, labels = [], []
    for videos, y in loader:
        logits = model(transform(videos).to(device))
        probs += torch.softmax(logits, dim=1)[:, 1].cpu().tolist()
        labels += y.tolist()
    return np.array(probs), np.array(labels)

def strip_report(model, loader, device):
    variants = {
        "normal": lambda x: x,
        "bottom 20% only (hood/dashboard)": keep_strip(bottom=0.2),
        "top 20% only (sky/buildings)": keep_strip(top=0.2),
        "left+right 20% only": keep_strip(sides=0.2),
    }
    for name, fn in variants.items():
        p, y = predict(model, loader, device, fn)
        f1 = f1_score(y, (p > 0.5).astype(int), average="macro", zero_division=0)
        print(f"{name:36s} F1 = {f1:.3f}   AUC = {roc_auc_score(y, p):.3f}   "
              f"predicted-positive rate = {(p > 0.5).mean():.2f}")


def stratified_report(probs, labels, val_df, meta_csv="nexar_data/metadata.csv"):
    meta = pd.read_csv(meta_csv)[["video_id", "light_conditions", "weather", "scene"]]
    d = val_df.reset_index(drop=True)[["video_id", "label"]].merge(meta, on="video_id", how="left")
    assert len(d) == len(probs), "val_df and loader order/length do not match"
    assert (d["label"].to_numpy() == labels).all(), "labels do not line up with val_df"

    d["prob"] = probs
    d["pred"] = (probs > 0.5).astype(int)

    print(f"Overall: F1 = {f1_score(d.label, d.pred, average='macro'):.3f}   "
          f"AUC = {roc_auc_score(d.label, d.prob):.3f}\n")

    for col in ["light_conditions", "scene", "weather"]:
        print(f"-- by {col} --")
        for g, s in d.groupby(col):
            if len(s) < 20 or s.label.nunique() < 2:
                print(f"  {g}: n={len(s)} (too small / one class, skipped)")
                continue
            print(f"  {g}: n={len(s)}, positive share={s.label.mean():.2f}, "
                  f"F1={f1_score(s.label, s.pred, average='macro'):.3f}, "
                  f"AUC={roc_auc_score(s.label, s.prob):.3f}")
        print()

In [16]:
strip_report(model, val_loader, device)
probs, labels = predict(model, val_loader, device)
stratified_report(probs, labels, val_df)

normal                               F1 = 0.641   AUC = 0.770   predicted-positive rate = 0.34
bottom 20% only (hood/dashboard)     F1 = 0.576   AUC = 0.634   predicted-positive rate = 0.41
top 20% only (sky/buildings)         F1 = 0.572   AUC = 0.625   predicted-positive rate = 0.29
left+right 20% only                  F1 = 0.657   AUC = 0.758   predicted-positive rate = 0.28
Overall: F1 = 0.641   AUC = 0.770

-- by light_conditions --
  Bright: n=1 (too small / one class, skipped)
  Dark: n=5 (too small / one class, skipped)
  Normal: n=151, positive share=0.49, F1=0.625, AUC=0.759
  Twilight: n=12 (too small / one class, skipped)

-- by scene --
  Highway: n=49, positive share=0.47, F1=0.550, AUC=0.654
  Other: n=2 (too small / one class, skipped)
  Rural: n=3 (too small / one class, skipped)
  Sub-urban: n=33, positive share=0.27, F1=0.618, AUC=0.898
  Urban: n=82, positive share=0.60, F1=0.617, AUC=0.754

-- by weather --
  Clear: n=89, positive share=0.46, F1=0.624, AUC=0.803
  C

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score

data["video_id"] = data["video_id"].astype(str)
meta = data.merge(df.assign(video_id=df["video_id"].astype(str))[["video_id","scene","weather","light_conditions"]],
                  on="video_id", how="left")
X = pd.get_dummies(meta[["scene","weather","light_conditions"]].fillna("NA"))
p = cross_val_predict(LogisticRegression(max_iter=1000), X, meta["label"],
                      cv=StratifiedKFold(5, shuffle=True, random_state=42), method="predict_proba")[:, 1]
print("metadata-only AUC:", roc_auc_score(meta["label"], p))

metadata-only AUC: 0.6290506639505055


In [10]:
# make sure the best checkpoint is loaded (the diagnostics cells may have changed the model)
checkpoint = torch.load("nexar_data/best_mvit_collision.pth", map_location=device, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

# weights, aligned to each split's own rows
w_map  = dict(zip(meta["video_id"], meta["w"]))
w_test = test_df["video_id"].astype(str).map(w_map).values
w_val  = val_df["video_id"].astype(str).map(w_map).values
print("NaN weights:", np.isnan(w_test).sum(), np.isnan(w_val).sum())   # should be 0 and 0

# test
test_probs, test_labels = get_probs(model, test_loader)
print(len(test_probs), len(test_labels), len(w_test))                   # all 170
print("TEST plain AUC   :", roc_auc_score(test_labels, test_probs))
print("TEST weighted AUC:", roc_auc_score(test_labels, test_probs, sample_weight=w_test))

# val (more data to judge by, since test is only 170 clips)
val_probs, val_labels = get_probs(model, val_loader)
print("VAL  plain AUC   :", roc_auc_score(val_labels, val_probs))
print("VAL  weighted AUC:", roc_auc_score(val_labels, val_probs, sample_weight=w_val))

NameError: name 'model' is not defined

In [7]:
# pip install transformers
from transformers import AutoModel

enc = AutoModel.from_pretrained("facebook/dinov2-base").to(device).eval()
if device.type == "cuda":
    enc = enc.half()
mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
std  = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

@torch.no_grad()
def clip_features(path):
    x = torch.from_numpy(np.load(path)).to(device).permute(0, 3, 1, 2).float() / 255   # T,3,H,W
    x = (x - mean) / std
    if device.type == "cuda":
        x = x.half()
    h = enc(pixel_values=x).last_hidden_state                  # T, 1+N, 768
    return torch.cat([h[:, 0], h[:, 1:].mean(1)], dim=1).float().cpu().numpy()   # T, 1536

all_df = pd.concat([train_df, val_df, test_df]).reset_index(drop=True)
feats = {}
for r in tqdm(all_df.itertuples(), total=len(all_df)):
    feats[str(r.video_id)] = clip_features(r.clip_path)
np.save("nexar_data/dinov2_feats.npy", feats, allow_pickle=True)

c:\Users\Toranj\Desktop\AIM\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Toranj\.cache\huggingface\hub\models--facebook--dinov2-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\Toranj\Desktop\AIM\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingfa

In [11]:
all_df = pd.concat([train_df, val_df, test_df]).reset_index(drop=True)
all_df["video_id"] = all_df["video_id"].astype(str)

meta = all_df[["video_id", "label"]].merge(
    df.assign(video_id=df["video_id"].astype(str))[["video_id", "scene", "weather", "light_conditions"]],
    on="video_id", how="left")

scene   = meta["scene"].where(meta["scene"].isin(["Highway", "Sub-urban", "Urban"]), "Other")
weather = meta["weather"].where(meta["weather"].isin(["Clear", "Cloudy"]), "Other")
light   = meta["light_conditions"].where(meta["light_conditions"] == "Normal", "NonNormal")
meta["cell"] = scene.astype(str) + "|" + weather.astype(str) + "|" + light.astype(str)

n_cell     = meta.groupby("cell")["label"].transform("size")
n_cell_lab = meta.groupby(["cell", "label"])["label"].transform("size")
meta["w"]  = (n_cell / (2 * n_cell_lab)).clip(upper=5)

w_map  = dict(zip(meta["video_id"], meta["w"]))
w_test = test_df["video_id"].astype(str).map(w_map).values
w_val  = val_df["video_id"].astype(str).map(w_map).values

print("NaN weights:", np.isnan(w_test).sum(), np.isnan(w_val).sum())   # should be 0 0

NaN weights: 0 0


In [12]:
class TemporalHead(nn.Module):
    def __init__(self, d_in=1536, d=256, T=16, layers=2, drop=0.3):
        super().__init__()
        self.proj = nn.Sequential(nn.LayerNorm(d_in * 2), nn.Dropout(drop), nn.Linear(d_in * 2, d))
        self.pos = nn.Parameter(torch.zeros(1, T, d))
        layer = nn.TransformerEncoderLayer(d, 4, d * 2, drop, batch_first=True, norm_first=True)
        self.tf = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
        self.out = nn.Sequential(nn.LayerNorm(d), nn.Dropout(drop), nn.Linear(d, 2))

    def forward(self, x):                                       # B,T,D
        diff = torch.cat([torch.zeros_like(x[:, :1]), x[:, 1:] - x[:, :-1]], dim=1)   # motion cue
        h = self.proj(torch.cat([x, diff], dim=-1)) + self.pos
        return self.out(self.tf(h).mean(1))

def fit_predict(tr_ids, tr_y, tr_w, pred_id_lists, epochs=25, seed=0, lr=3e-4):
    set_seed(seed)
    X = torch.tensor(np.stack([feats[i] for i in tr_ids]))
    Y = torch.tensor(tr_y, dtype=torch.long)
    W = torch.tensor(tr_w, dtype=torch.float32)
    model_t = TemporalHead().to(device)
    opt = AdamW(model_t.parameters(), lr=lr, weight_decay=0.05)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    for ep in range(epochs):
        model_t.train()
        perm = torch.randperm(len(X))
        for i in range(0, len(X), 32):
            b = perm[i:i + 32]
            xb = X[b].to(device)
            xb = xb + 0.05 * xb.std() * torch.randn_like(xb)            # feature noise
            loss = (F.cross_entropy(model_t(xb), Y[b].to(device), reduction="none", label_smoothing=0.1)
                    * W[b].to(device)).mean() / W[b].mean().to(device)
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step()
    model_t.eval()
    outs = []
    with torch.no_grad():
        for ids_ in pred_id_lists:
            Xp = torch.tensor(np.stack([feats[i] for i in ids_])).to(device)
            outs.append(torch.softmax(model_t(Xp), 1)[:, 1].cpu().numpy())
    return outs

# ---- 5-fold CV on train+val; test set untouched until the end ----
trainval = pd.concat([train_df, val_df]).reset_index(drop=True)
ids = trainval["video_id"].astype(str).values
y   = trainval["label"].values
w   = np.array([w_map[i] for i in ids])
test_ids = test_df["video_id"].astype(str).values

oof, test_preds = np.zeros(len(trainval)), []
for fold, (tr, va) in enumerate(StratifiedKFold(5, shuffle=True, random_state=42).split(ids, y)):
    pv, pt = fit_predict(ids[tr], y[tr], w[tr], [ids[va], test_ids], seed=fold)
    oof[va] = pv
    test_preds.append(pt)
    print(f"fold {fold}: AUC {roc_auc_score(y[va], pv):.3f}")

print("OOF AUC          :", roc_auc_score(y, oof))
print("OOF weighted AUC :", roc_auc_score(y, oof, sample_weight=w))
tp = np.mean(test_preds, axis=0)
print("TEST AUC         :", roc_auc_score(test_df["label"], tp))
print("TEST weighted AUC:", roc_auc_score(test_df["label"], tp, sample_weight=w_test))

fold 0: AUC 0.736
fold 1: AUC 0.676
fold 2: AUC 0.706
fold 3: AUC 0.700
fold 4: AUC 0.624
OOF AUC          : 0.6834218818781298
OOF weighted AUC : 0.6504340677643905
TEST AUC         : 0.6672664359861591
TEST weighted AUC: 0.6286989848321936
